# 🔬 AI-Aided Identification of Polymer Waste
### Workshop Notebook — Attendee Version
*Dr Benji Fenech-Salerno, Imperial College London*


Workshop delivered by Dr Benji Fenech-Salerno & Dr Rebecca L. Jones for the AIchemy Chemistry Education 2nd Annual GenAI in Chemistry Education Conference. 

---

## The problem

Only about **9% of plastic waste is recycled globally**. A major bottleneck is *sorting*: different polymers (PET, HDPE, polypropylene…) must be separated before they can be reprocessed, and telling them apart by eye is almost impossible.

**Near-infrared (NIR) spectroscopy** offers a fast, non-destructive fix. Each polymer absorbs NIR light at characteristic wavelengths, producing a unique spectral "fingerprint". Pair that fingerprint with a **machine-learning classifier** and the sorting can be automated.

In this notebook you will build that classifier and use it to identify five mystery samples.


## How to use this notebook


Run every cell **in order** with **Shift + Enter** (or the ▶ button).

> **This is the attendee workbook — you write the code.**
> **Step 0 is ready to run.** From Step 1 onward, each code cell gives you the goal,
> numbered `# TODO` steps, and a 💬 prompt you can paste into Claude / ChatGPT.
> Fill in the code yourself (with GenAI's help), then run the cell. Values marked 🔧
> are already written — just change them. Extension cells (🚀) have **no starter code** —
> they're open challenges if you finish early.

There are **two tracks**:

| Track | Icon | Who | What |
|-------|------|-----|------|
| **Core** | ⭐ | Everyone | The essential path. Complete these and you'll finish within the hour. |
| **Extension** | 🚀 | If you're ahead | Optional deeper dives. Skip them if you're short on time — nothing later depends on them. |

Other icons you'll see inside cells:

| Icon | Meaning |
|------|---------|
| 🔧 | A value to change — edit it and re-run the cell |
| 💬 | A prompt to try in Claude / ChatGPT |
| ❓ | A discussion question |

> **Files needed in the same folder as this notebook**
> - `polymers_workshop.csv` — training data (75 spectra, 5 polymer types)
> - `polymers_unknowns_student.csv` — five unlabelled mystery samples

---
## ⚙️ Step 0 — Setup ⭐

### 0a. Install the packages

Run the cell below **first**. If you're using **JupyterLite** (running Python in your browser), this downloads the libraries we need. The **first run can take 30–60 seconds** — wait for the ✅ before continuing.


In [ ]:
# 📦 Install packages needed for this workshop.
# First run in JupyterLite is slow (downloading into the browser) — please wait for ✅.
try:
    import piplite
    await piplite.install(['numpy', 'pandas', 'matplotlib', 'scikit-learn'])
    print("✅ Packages installed (JupyterLite / browser).")
except ImportError:
    # Not in JupyterLite (e.g. Anaconda / Colab / local Jupyter) — packages already present.
    print("✅ Standard Jupyter detected — packages already available.")

### 0b. Import the libraries

Now load everything into memory.

- **NumPy** / **Pandas** — numerical data and tables
- **Matplotlib** — plotting
- **scikit-learn** — the machine-learning classifier


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

print("✅ All libraries loaded.")

---
## 📂 Step 1 — Load the data ⭐

The dataset holds **75 NIR spectra** from five polymer types, measured on a handheld PlasTell NIR instrument.

Each spectrum records reflected near-infrared light across **1550–1950 nm** (128 wavelength points). One row of the CSV = one spectrum.

The five polymers were chosen on purpose:

| Group | Polymers | Why interesting? |
|-------|----------|------------------|
| Similar pair 1 | **HDPE** & **LDPE** | Both polyethylenes — nearly identical spectra |
| Similar pair 2 | **PET** & **PC** | Both contain C=O groups — moderately similar |
| Outlier | **PA66** | Nylon — chemically distinct, easy to separate |


In [ ]:
# --- Load the data ---
# GOAL: read the two CSV files and pull out the pieces you'll need later.
#
# TODO 1. Read 'polymers_workshop.csv' into a pandas DataFrame.
# TODO 2. From it, create three variables:
#           labels       -> the 'polymer' column as an array   (the class of each spectrum)
#           wavelengths  -> the column names EXCEPT 'polymer', converted to float
#           spectra      -> the numeric spectral values as an array   (shape 75 x 128)
# TODO 3. Read 'polymers_unknowns_student.csv' and create:
#           unknown_names    -> the 'sample' column
#           unknown_spectra  -> ONLY the numeric spectral columns (drop the text columns)
#      NOTE: the true identity of each unknown is not for you to read here — you'll
#            predict them in Step 5, then discuss with a demonstrator.
#
# 💬 Ask GenAI: "How do I load a CSV with pandas and split one column off as labels?"

# ----- given for you: the five classes and a consistent colour per polymer -----
POLYMERS = ['HDPE', 'LDPE', 'PET', 'PC', 'PA66']
COLORS   = {'HDPE': '#2196F3', 'LDPE': '#90CAF9',    # blues — similar pair 1
            'PET':  '#E53935', 'PC':   '#EF9A9A',    # reds  — similar pair 2
            'PA66': '#43A047'}                        # green — outlier

# TODO 4. print a short summary: number of spectra, the wavelength range, the unknown names.

---
## 🔎 Step 2 — Explore the data ⭐

Before any machine learning, get a feel for the data. Two quick questions:

1. **How many spectra do we have per polymer?**
2. **How similar are the polymers to each other?** (This predicts which ones the classifier will confuse.)


In [ ]:
# --- 2a. How many spectra per class? ---
# GOAL: for each polymer in POLYMERS, count how many spectra belong to it.
#
# TODO: loop over POLYMERS, count the matches in `labels`
#       (hint: np.sum(labels == polymer)), and print each polymer with its count.
#
# 💬 Ask GenAI: "How do I count how many times each value appears in a numpy array?"

In [ ]:
# --- 2b. How similar are the classes to each other? ---
# GOAL: build the AVERAGE spectrum of each polymer, then measure how alike they are.
#
# TODO 1. For each polymer, compute its mean spectrum (average of its rows in `spectra`).
# TODO 2. Put the mean spectra into a DataFrame and call .corr() for the pairwise
#         correlation (r). Print it rounded to ~3 decimals.
# TODO 3. Which pair is closest to r = 1.0? That's the pair the classifier will struggle with.
#
# 💬 Ask GenAI: "How do I compute a correlation matrix between columns with pandas?"

❓ **Discuss**

1. Which two polymers have the most similar average spectra? Does that match the chemistry (look back at the table in Step 1)?
2. Which polymer stands out as most different — and what does that predict about how easily the classifier will identify it?


### 🚀 Extension — within-class variability

*Optional.* How **consistent** are the spectra inside each class? The **coefficient of variation (CV)** = std ÷ mean. A high CV means noisy, variable measurements.


In [ ]:
# 🚀 EXTENSION — optional, no starter code. Build it yourself with GenAI.
#
# Challenge: show how VARIABLE each polymer's spectra are.
# For each class compute the coefficient of variation (CV = std / mean × 100%)
# across all of its spectra, and print a small table. Which class is most variable?
#
# 💬 Ask GenAI: "For each group in my data, compute the coefficient of variation and print a table."

---
## 📈 Step 3 — Visualise the spectra ⭐

Numbers tell part of the story; a picture tells the rest. Plot the **average spectrum ± 1 standard deviation** for each polymer. The shaded band shows within-class variability — a wide band means noisy measurements.


In [ ]:
# --- 3a. Average spectrum ± 1 SD per polymer ---
# GOAL: one line per polymer showing its mean spectrum, with a shaded ±1 SD band.
#
# TODO 1. Create a matplotlib figure and axes.
# TODO 2. For each polymer: compute the mean and std across its spectra; plot the mean
#         against `wavelengths`, then use ax.fill_between(...) for the ±1 SD band.
#         Use COLORS[polymer] so the colours match the rest of the notebook.
# TODO 3. Label the axes, add a legend, and show the plot.
#
# 💬 Ask GenAI: "How do I use matplotlib fill_between to shade a band around a line?"

❓ **Discuss**

1. Can you *see* the difference between HDPE and LDPE? Between PET and PC? Does PA66 look obviously different?
2. If you were a recycling engineer with no computer, could you sort these by eye? What does that tell you about why ML is useful here?

💬 **Ask GenAI**
> *"HDPE and LDPE are both polyethylenes. In plain English, why are their NIR spectra so similar, and what structural differences might create the small differences that remain?"*


### 🚀 Extension — the whole dataset as a heatmap

*Optional.* Every spectrum as one row, colour = intensity. Spectra are sorted by polymer so you can see each class as a block.


In [ ]:
# 🚀 EXTENSION — optional, no starter code. Build it yourself with GenAI.
#
# Challenge: draw a heatmap of all 75 spectra (each row = one spectrum, colour = intensity),
# with the spectra grouped/sorted by polymer class so each class shows up as a block.
#
# 💬 Ask GenAI: "How do I show a 2D numpy array as a heatmap with matplotlib imshow?"

---
## 🤖 Step 4 — Build the classifier ⭐

### How k-Nearest Neighbours (kNN) works

To identify a new spectrum, kNN:
1. measures the **distance** from it to every training spectrum,
2. finds the **k closest** ("nearest neighbours"),
3. takes a **majority vote** among them.

**Why normalise first?** Raw intensities span 0–65,535, so a "brighter" measurement could dominate. Scaling each spectrum to 0–1 makes the classifier focus on **spectral shape**, not brightness — standard NIR practice.

You'll do four things: **normalise → split → train → evaluate.**


In [ ]:
# --- 4a. Normalise each spectrum to [0, 1] ---
# WHY: raw spectra differ in overall brightness. Scaling each spectrum to its own
#      min–max makes the classifier compare SHAPE rather than brightness.
#
# TODO 1. Write a function normalise(X) that scales each ROW of X to the range [0, 1].
#         (hint: use X.min(axis=1, keepdims=True) and X.max(...); add a tiny value to avoid ÷0)
# TODO 2. Apply it:   X_norm       = normalise(spectra)
#                     unknown_norm = normalise(unknown_spectra)
#
# 💬 Ask GenAI: "Write a numpy function that min-max scales each row of a 2D array to 0..1."

In [ ]:
# --- 4b. Split into training and test sets ---
# GOAL: hold back some spectra so we can test the model on data it never saw.

# 🔧 Fraction of the data kept for testing (change this later and watch what happens):
TEST_SIZE = 0.30

# TODO: use train_test_split(...) on X_norm and labels to create
#       X_train, X_test, y_train, y_test.
#       Pass test_size=TEST_SIZE, random_state=42, and stratify=labels
#       (stratify keeps each class balanced across both sets). Then print the two set sizes.
#
# 💬 Ask GenAI: "What does the stratify argument do in sklearn's train_test_split?"

In [ ]:
# --- 4c. Train the kNN classifier ---

# 🔧 How many neighbours get to vote (try 1, 3, 7... later):
K = 3

# TODO: create a KNeighborsClassifier with n_neighbors=K, then .fit(...) it on
#       X_train and y_train. Store the model as `knn`, and print a confirmation.
#
# 💬 Ask GenAI: "How does a k-nearest-neighbours classifier decide on a prediction?"

In [ ]:
# --- 4d. Evaluate on the held-out test set ---
# TODO 1. Predict the test spectra:   y_pred = knn.predict(X_test)
# TODO 2. Compare with the truth:      accuracy = accuracy_score(y_test, y_pred)
# TODO 3. print the accuracy as a percentage.
#
# 💬 Ask GenAI: "Why is it misleading to measure accuracy on the same data used to train?"

### The confusion matrix ⭐

Accuracy alone hides *which* mistakes happen. A **confusion matrix** shows it: rows = true polymer, columns = predicted. The diagonal is correct; anything off-diagonal is a mix-up.


In [ ]:
# --- 4e. Confusion matrix ---
# A confusion matrix shows WHICH polymers get mixed up (rows = true, columns = predicted).
#
# TODO 1. Build it:   cm = confusion_matrix(y_test, y_pred, labels=POLYMERS)
# TODO 2. Display it: ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=POLYMERS).plot(...)
#         Add a title. The bright diagonal = correct; off-diagonal = confusions.
# TODO 3. Which pair is confused most? Does it match the correlation you found in Step 2?
#
# 💬 Ask GenAI: "How do I build and plot a confusion matrix with scikit-learn?"

❓ **Discuss**

1. Which polymer pair (if any) gets confused? Does it match the high correlation you found in Step 2?
2. Why would it be *wrong* to test the model on the same spectra it was trained on?

💬 **Ask GenAI**
> *"Explain what `train_test_split` does in scikit-learn, and why testing a model on its own training data gives a misleading accuracy."*


### 🚀 Extension — tune k (a hyperparameter sweep)

*Optional.* There's no single "right" k. Loop over many values and plot the accuracy of each — this is how real models are tuned.


In [ ]:
# 🚀 EXTENSION — optional, no starter code. Build it yourself with GenAI.
#
# Challenge: loop k from 1 to 15, train a kNN for each value, record the test accuracy,
# and plot accuracy vs k. Which k is best? Why might a very LARGE k hurt performance?
#
# 💬 Ask GenAI: "How do I tune the number of neighbours k in kNN and plot accuracy against k?"

---
## 🕵️ Step 5 — Identify the mystery samples ⭐

Five unlabelled samples came off a real recycling line. Your trained classifier will guess each one — then you'll reveal the truth. This is exactly the real-world task: scan an unknown plastic and let the model classify it.

Two of these samples are deliberately tricky. Keep an eye on what the classifier does with a **mixture** and with a material it was **never trained on** — this is where the limits of machine learning show up.

In [ ]:
# --- 5a. Classify the five unknown samples ---
# TODO 1. Predict a polymer for each unknown:   predictions   = knn.predict(unknown_norm)
# TODO 2. Get the vote share (confidence):       probabilities = knn.predict_proba(unknown_norm)
# TODO 3. print each unknown's name, its predicted polymer, and its confidence
#         (the confidence is the largest value in that sample's probability row).
#
# 💬 Ask GenAI: "What does predict_proba return for a kNN classifier, and how do I read it?"

In [ ]:
# --- 5b. Check your results ---
# The true identities of the mystery samples are NOT in your data — on purpose!
#
# Look back at your predictions and confidence scores from 5a and think about:
#   • Which sample was the model most — and least — confident about?
#   • Are any of the unknowns near a class boundary you spotted in Step 2?
#   • Do any of the predictions look surprising or suspicious to you?
#
# 🙋 Now call over one of the demonstrators and talk through your predictions,
#    your confidence scores, and your confusion matrix before you finish.

❓ **Discuss**

1. Which unknown was predicted most confidently? Does that fit the correlations from Step 2?
2. If a sample was misclassified, is that the fault of the *classifier*, the *data*, or the *measurement*?
3. Unknown E (PDMS) is a silicone that was never in the training data, yet the model gave a confident answer. Why can't a standard classifier say *"none of these"*? How might you detect an out-of-distribution sample in practice?

💬 **Ask GenAI**
> *"HDPE and LDPE give nearly identical NIR spectra. Propose two ways — a different analytical technique or different data preprocessing — that might let a classifier tell them apart. Then: how would I check whether those suggestions are chemically valid?"*

### 🚀 Extension — see *why* the model voted the way it did

*Optional.* Plot each unknown against the average spectrum of its predicted class, with the vote breakdown as an inset bar chart.


In [ ]:
# 🚀 EXTENSION — optional, no starter code. Build it yourself with GenAI.
#
# Challenge: for each unknown, plot its spectrum against the average spectrum of the class
# the model predicted, and show the vote share for every class as a bar chart alongside it.
#
# 💬 Ask GenAI: "For each test sample, plot its spectrum next to its predicted class mean,
#    plus a bar chart of the class probabilities from predict_proba."

---
## 🎯 Summary

You built a complete NIR polymer-identification pipeline: **load → explore → visualise → train → deploy on unknowns** — the same workflow used in the ID1 lab.

**Four takeaways**

1. **Explore first.** The correlation matrix (Step 2) predicted the HDPE/LDPE problem *before* any ML.
2. **Accuracy isn't everything.** The confusion matrix shows *which* errors happen — and in recycling, some mix-ups matter more than others.
3. **The limit is often the data.** A perfect algorithm can't separate polymers whose spectra genuinely overlap.
4. **In ID1 you're assessed on *explanation*, not accuracy.** Can you explain *why* the confusion matrix looks the way it does, and *why* changing k changes accuracy?

---

## 🚀 If you finished early — try these

- **Retune the model:** change `K` (Step 4c) and `TEST_SIZE` (Step 4b) and watch the confusion matrix respond.
- 💬 **Ask GenAI to swap the algorithm:** *"Replace this kNN classifier with a scikit-learn Random Forest on the same normalised data, and compare the confusion matrices."*
- 💬 **Ask GenAI to grow the dataset:** *"How would I add a 6th polymer class (e.g. polypropylene) to this pipeline?"*
- **Reflect:** if this ran on a real recycling line, would you tune it to avoid *false positives* or *false negatives*? What's the trade-off?

---

*Workbook created by: Dr Benji Fenech-Salerno, b.fenech-salerno@imperial.ac.uk, Imperial College London.*
